# Cleaning and Combining

Trap counts (the A2 data) and the weather at The Forks. Each step: what, why, cost, check.

In [1]:
import pandas as pd
import mosquito_tools as tools

from pathlib import Path

# Settings: change these, not the code below
TRAP_FILES = ["../earlier-work/a1-mosquitoes/data/raw/traps-2015-2025.csv",   # the A1 export, unchanged
              *sorted(Path("../data/raw/traps").glob("*.csv"))]           # every newer season
WEATHER_FILES = sorted(Path("../data/raw/weather").glob("*.csv"))
SPRING_MONTHS = [5, 6]          # May and June: the rain that fills the ponds
DAILY_FILE = "../data/clean/daily.csv"
SEASONS_FILE = "../data/clean/seasons.csv"

In [2]:
traps_raw = tools.load_releases(TRAP_FILES, ["Count Date"])
weather_raw = tools.load_releases(WEATHER_FILES, ["Date/Time"])

1 file(s), 880 rows
11 file(s), 4018 rows


**Step 1: dates, day first.** What: parse the trap dates as day first. Why: the file writes `13/05/2015`; read
month first, it fails or swaps days and months. Cost: none. Check: every count date is May to October.

In [3]:
traps = traps_raw.rename(columns={"City Wide Daily Average": "city_avg", "North West Average": "north_west",
                                  "North East Average": "north_east", "South East Average": "south_east",
                                  "South West Average": "south_west"})
traps["date"] = pd.to_datetime(traps["Count Date"], dayfirst=True)
traps = traps[["date", "city_avg", "north_west", "north_east", "south_east", "south_west"]]
assert traps["date"].dt.month.between(5, 10).all()
assert tools.count_repeats(traps, ["date"]) == 0
traps["date"].dt.year.value_counts().sort_index()

date
2015    102
2016     86
2017     61
2018     61
2019     77
2020     76
2021     60
2022    118
2023     73
2024    107
2025     59
Name: count, dtype: int64

**Step 2: the weather, three columns.** What: keep the date, mean temperature and total precipitation. Why: that's
all the story uses. Cost: the other columns (snow, degree days) aren't checked. Check: one row per day.

In [4]:
weather = pd.DataFrame({
    "date": pd.to_datetime(weather_raw["Date/Time"]),
    "mean_temp": weather_raw["Mean Temp (°C)"],
    "precip_mm": weather_raw["Total Precip (mm)"],
})
assert tools.count_repeats(weather, ["date"]) == 0
print(weather["date"].min().date(), "to", weather["date"].max().date())

2015-01-01 to 2025-12-31


**Step 3: join each count day to that day's weather.** Every count day should find one.

In [5]:
daily = traps.merge(weather, on="date", how="left", indicator="source")
missing = tools.unmatched(traps, weather, "date")
assert missing == [], missing
daily["source"].value_counts()

source
both          880
left_only       0
right_only      0
Name: count, dtype: int64

**Step 4: one row per summer.** What: the season average per count day, the number of count days, the peak, and May
and June rain. Why: count days differ from summer to summer, so totals aren't fair; the story compares summers. Cost:
missing rain days count as zero in the spring total (reported below). Check: every summer has a rain total.

In [6]:
seasons = daily.assign(year=daily["date"].dt.year).groupby("year").agg(
    season_avg=("city_avg", "mean"), count_days=("date", "size"), peak=("city_avg", "max")).reset_index()
peak_rows = daily.loc[daily.groupby(daily["date"].dt.year)["city_avg"].idxmax()]
seasons["peak_date"] = peak_rows["date"].to_numpy()

spring_days = weather[weather["date"].dt.month.isin(SPRING_MONTHS)]
spring = spring_days.groupby(spring_days["date"].dt.year).agg(
    spring_precip_mm=("precip_mm", "sum"), spring_days_missing=("precip_mm", lambda s: int(s.isna().sum())))
spring = spring.rename_axis("year").reset_index()

missing_years = tools.unmatched(seasons, spring, "year")
assert missing_years == [], missing_years
seasons = seasons.merge(spring, on="year", how="inner")
seasons.round(1)

,year,season_avg,count_days,peak,peak_date,spring_precip_mm,spring_days_missing
0,2015,22.9,102,151,2015-06-11,168.2,5
1,2016,28.3,86,187,2016-06-17,170.9,3
2,2017,2.6,61,19,2017-07-10,96.1,0
3,2018,4.2,61,20,2018-07-11,149.6,0
4,2019,15.5,77,108,2019-07-28,63.6,0
5,2020,16.2,76,101,2020-06-30,114.8,0
6,2021,6.0,60,42,2021-07-02,93.7,0
7,2022,25.5,118,235,2022-08-29,231.0,0
8,2023,6.7,73,53,2023-06-21,62.1,0
9,2024,16.0,107,138,2024-07-13,227.4,0


In [7]:
daily.drop(columns="source").to_csv(DAILY_FILE, index=False)
seasons.to_csv(SEASONS_FILE, index=False)
assert len(pd.read_csv(SEASONS_FILE)) == len(seasons) and len(pd.read_csv(DAILY_FILE)) == len(daily)
print(len(daily), "count days,", len(seasons), "summers saved")

880 count days, 11 summers saved
